# Module 25 — Learning Rate Schedules

Every training loop so far used a fixed (or simply linearly-decaying,
Module 03) learning rate. Real transformer training uses a **schedule**
with two phases:

1. **Linear warmup**: for the first few hundred/thousand steps, the
   learning rate ramps up from 0 to its peak value, instead of starting at
   full strength immediately.
2. **Cosine decay**: after warmup, the learning rate smoothly decreases
   from its peak toward (near) 0 over the rest of training, following a
   cosine curve.

This module implements the schedule from scratch, verifies it against a
standard library implementation, and demonstrates concretely *why* warmup
matters — not just asserts it.

## 1. The schedule, implemented from scratch

In [ ]:
import math

def lr_multiplier(step, warmup_steps, total_steps):
    """Returns a multiplier in [0, 1] to scale the peak learning rate by."""
    if step < warmup_steps:
        return step / max(1, warmup_steps)
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    progress = min(progress, 1.0)
    return 0.5 * (1.0 + math.cos(math.pi * progress))


warmup_steps, total_steps, peak_lr = 10, 100, 1e-3
schedule = [peak_lr * lr_multiplier(s, warmup_steps, total_steps) for s in range(total_steps)]
print("lr at step 0:  ", schedule[0])
print("lr at step 5:  ", schedule[5])
print("lr at step 10: ", schedule[10], "(peak)")
print("lr at step 99: ", schedule[99])

## 2. Verifying against Hugging Face's `get_cosine_schedule_with_warmup`

This is the exact same formula (with its default `num_cycles=0.5`) used
throughout the Hugging Face ecosystem — a real, widely-used reference
implementation, not just something we hoped matched.

In [ ]:
import torch
from transformers import get_cosine_schedule_with_warmup

model = torch.nn.Linear(4, 4)
optimizer = torch.optim.AdamW(model.parameters(), lr=peak_lr)
hf_scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)

hf_lrs = []
for step in range(total_steps):
    hf_lrs.append(optimizer.param_groups[0]["lr"])
    optimizer.step()
    hf_scheduler.step()

for step in range(total_steps):
    assert abs(schedule[step] - hf_lrs[step]) < 1e-9, f"mismatch at step {step}: {schedule[step]} vs {hf_lrs[step]}"
print("From-scratch schedule matches transformers\' get_cosine_schedule_with_warmup exactly, all 100 steps.")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 4))
plt.plot(schedule)
plt.axvline(warmup_steps, color="gray", linestyle="--", label="end of warmup")
plt.xlabel("step")
plt.ylabel("learning rate")
plt.title("Warmup + cosine decay")
plt.legend()
plt.show()

## 3. Why warmup matters: a real instability demonstration

Adam's moment estimates (Module 24) are noisy and biased in the very
first steps, and the model's weights start essentially random — taking a
full-strength step immediately can be actively destructive. Using a
deliberately aggressive peak learning rate to make the effect obvious:
compare training with vs. without warmup, tracking the **worst** loss seen
in the first 30 steps as a measure of early instability.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

def make_model():
    torch.manual_seed(1)
    return nn.Sequential(
        nn.Linear(16, 128), nn.ReLU(),
        nn.Linear(128, 128), nn.ReLU(),
        nn.Linear(128, 128), nn.ReLU(),
        nn.Linear(128, 10),
    )

x = torch.randn(32, 16)
y = torch.randint(0, 10, (32,))
aggressive_peak_lr = 1.0  # deliberately too high, to stress-test this
warmup_steps_demo = 30

def lr_for(step, use_warmup):
    if use_warmup and step < warmup_steps_demo:
        return aggressive_peak_lr * (step + 1) / warmup_steps_demo
    return aggressive_peak_lr

results = {}
for use_warmup in [False, True]:
    model = make_model()
    optimizer = torch.optim.AdamW(model.parameters(), lr=aggressive_peak_lr)
    losses = []
    for step in range(100):
        for group in optimizer.param_groups:
            group["lr"] = lr_for(step, use_warmup)
        optimizer.zero_grad()
        loss = F.cross_entropy(model(x), y)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    results[use_warmup] = losses
    print(f"warmup={use_warmup!s:>5}   worst loss in first 30 steps: {max(losses[:30]):>12.1f}   final loss: {losses[-1]:.3f}")

assert max(results[False][:30]) > max(results[True][:30]) * 100
print("\nConfirmed: without warmup, this aggressive learning rate causes the loss to spike orders of magnitude higher early in training.")

## Recap

- Warmup + cosine decay was implemented from scratch and verified to match
  Hugging Face's `get_cosine_schedule_with_warmup` exactly, step for step.
- The concrete reason warmup matters: with a deliberately aggressive peak
  learning rate, training without warmup spiked to a loss orders of
  magnitude worse in the first 30 steps than the same run with warmup —
  a real, measured instability, not a hypothetical one.
- Both runs eventually reached a similar final loss here (a small toy
  problem is forgiving), but real large-scale pretraining runs (Module 31)
  often don't recover from this kind of early instability at all —
  warmup is standard practice specifically to avoid ever finding out.

Module 26 covers the other two standard training-stability techniques:
gradient clipping and weight decay.